In [1]:
import pandas as pd
import re
import glob

def load_and_preprocess(csv_path):
    """读取单个CSV并提取日期、年、月"""
    df = pd.read_csv(csv_path)

    df['date'] = df['scene_name'].str.extract(r'_(\d{8})T')
    df['date'] = pd.to_datetime(df['date'], format="%Y%m%d", errors="coerce")

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

    return df


def compute_monthly_stats(df):
    """生成与你论文一致的月度 floe + lead 统计表格"""

    summary = df.groupby("month").agg(

        # Floe
        CS2_Floe_Density=('density_CS2_floe', 'mean'),
        CS2_Floe_Std    =('density_CS2_floe', 'std'),
        S1_Floe_Density =('density_S1_floe', 'mean'),
        S1_Floe_Std     =('density_S1_floe', 'std'),

        # Lead
        CS2_Lead_Density=('density_CS2_lead_only', 'mean'),
        CS2_Lead_Std    =('density_CS2_lead_only', 'std'),
        S1_Lead_Density =('density_S1_lead_only', 'mean'),
        S1_Lead_Std     =('density_S1_lead_only', 'std'),
    ).reset_index()

    return summary


In [ ]:

file_path = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\2023_density_simplest_filter\overall_density_summary_2023_raster_filtered_detailed.csv"
df = load_and_preprocess(f)
year = df["year"].iloc[0]

print(f"Processing year: {year}")

yearly_results[year] = compute_monthly_stats(df)

In [ ]:
input_folder = r"D:\SeaIce\overlap\*.csv"  # 修改为你的文件夹路径

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
# File path (Replace with your actual file path)
file_path = r"C:\Users\TJ002\Desktop\CS2_S1_result\overlap\2023_density_simplest_filter\overall_density_summary_2023_raster_filtered_detailed.csv"

files = glob.glob(input_folder)

yearly_results = {}
all_df = []

for f in files:
    df = load_and_preprocess(f)
    year = df["year"].iloc[0]

    print(f"Processing year: {year}")

    yearly_results[year] = compute_monthly_stats(df)
    all_df.append(df)

# 合并所有年份
merged_df = pd.concat(all_df, ignore_index=True)
merged_summary = compute_monthly_stats(merged_df)

print("All years processed.")
